# manga-lens: fine-tune nhận dạng nhân vật (face / body)
**Chỉ cần Run All một lần.** Bật trước: **Internet On**, **GPU T4**, Secret `HF_TOKEN` (Attached).
Quy trình: train trên 87 bộ **train** → chọn cấu hình face **tự động theo val** (BCubed F1) → chạy **test đúng một lần** cho cấu hình đã chọn.
Lệnh nào lỗi thì notebook dừng ngay (không chạy tiếp với kết quả hỏng). Ước tính tổng ~1,5–2,5 giờ (chưa kiểm chứng); crop cắt một lần rồi cache.
Mỗi phiên mới phải chạy lại từ đầu; chỉ `/kaggle/working` được giữ.

In [ ]:
import os, subprocess, json, shutil, glob
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PYTHONUTF8"] = "1"

def sh(cmd, cwd=None):
    """Chạy lệnh shell, in log trực tiếp, DỪNG notebook nếu lệnh lỗi."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): {cmd}")

sh("pip -q install huggingface_hub")

## 1. Tải Manga109 (4,1 GB, vài phút)

In [ ]:
M109 = "/kaggle/working/m109/Manga109_released_2026_05_21"
if os.path.isdir(M109 + "/annotations") and os.path.isdir(M109 + "/images"):
    print("Manga109 đã có sẵn, bỏ qua tải/giải nén")
else:
    sh("hf download hal-utokyo/Manga109 Manga109_released_2026_05_21.zip --repo-type dataset --local-dir /kaggle/working/m109 "
       "|| huggingface-cli download hal-utokyo/Manga109 Manga109_released_2026_05_21.zip --repo-type dataset --local-dir /kaggle/working/m109")
    sh("cd /kaggle/working/m109 && unzip -qo Manga109_released_2026_05_21.zip && rm Manga109_released_2026_05_21.zip")
sh("ls /kaggle/working/m109 " + M109)

## 2. Code từ GitHub + nhãn nhân vật
Nếu `ls` ở trên cho thấy `images/` và `annotations/` nằm trong thư mục con khác, sửa `ROOT`.

In [ ]:
sh("[ -d manga-lens ] && git -C manga-lens pull || git clone https://github.com/CPhuongg/manga-lens.git")
import sys
sys.path.insert(0, "manga-lens/scripts")
from pathlib import Path
from convert_manga109 import split_titles   # cùng cách chia (seed 0, theo bộ) như lúc train detector
ROOT = "/kaggle/working/m109/Manga109_released_2026_05_21"
titles = [p.stem for p in Path(ROOT, "annotations").glob("*.xml")]
sp = split_titles(titles, seed=0)
os.makedirs("/kaggle/working/titles", exist_ok=True)
for s, t in sp.items():
    Path(f"/kaggle/working/titles/{s}_titles.txt").write_text(chr(10).join(sorted(t)))
    print(s, len(t), "bộ")
assert [len(sp[s]) for s in ("train", "val", "test")] == [87, 11, 11], "chia split lệch so với lúc train detector"
sh(f"python scripts/extract_characters.py --root {ROOT} --yolo /kaggle/working/titles --out /kaggle/working/characters.json", cwd="manga-lens")

## 3. Thử nhanh (2 bộ, vài bước) — lỗi pipeline sẽ lộ ra ở đây trước khi tốn thời gian

In [ ]:
CACHE = "/kaggle/temp/crops" if os.path.isdir("/kaggle/temp") else "/kaggle/working/crops"
CH = "/kaggle/working/characters.json"
OUT = "/kaggle/working/out/reid"
COMMON = f"--root {ROOT} --chars {CH} --cache {CACHE} --out {OUT}"
sh(f"python scripts/train_reid.py --root {ROOT} --chars {CH} --cache {CACHE}_smoke --out /kaggle/working/out/smoke "
   f"--cls face --tag smoke --epochs 1 --iters 10 --max-books 2", cwd="manga-lens")
shutil.rmtree(f"{CACHE}_smoke", ignore_errors=True)

## 4. Face: 3 cấu hình, train trên TRAIN, theo dõi val mỗi epoch

In [ ]:
FACE_CFG = {
    "f_a": "--unfreeze 4  --lr-bb 2e-5",
    "f_b": "--unfreeze 12 --lr-bb 1e-5",
    "f_c": "--unfreeze 4  --lr-bb 2e-5 --temp 0.05",
}
for tag, extra in FACE_CFG.items():
    print("=" * 20, tag, extra)
    sh(f"python scripts/train_reid.py {COMMON} --cls face --tag {tag} {extra} --epochs 10", cwd="manga-lens")

## 5. Chọn cấu hình face TỰ ĐỘNG theo val (BCubed F1; không đụng test)

In [ ]:
val_res = {}
for tag in FACE_CFG:
    print("=" * 20, tag)
    sh(f"python scripts/eval_cluster.py --chars {CH} --tag {tag} --feat face --emb-dir {OUT} --no-test --json {OUT}/{tag}_val.json", cwd="manga-lens")
    val_res[tag] = json.load(open(f"{OUT}/{tag}_val.json"))
BEST_FACE = max(val_res, key=lambda t: val_res[t]["val_bcF"])
print()
print("VAL bcF:", {t: round(r["val_bcF"], 3) for t, r in val_res.items()}, "-> chọn", BEST_FACE)

## 6. Test face — một lần, cho cấu hình đã chọn

In [ ]:
sh(f"python scripts/train_reid.py {COMMON} --cls face --tag {BEST_FACE} --eval-only", cwd="manga-lens")
sh(f"python scripts/eval_cluster.py --chars {CH} --tag {BEST_FACE} --feat face --emb-dir {OUT} --json {OUT}/{BEST_FACE}_test.json", cwd="manga-lens")

## 7. Body (một cấu hình)
Xoá cache crop face trước để đỡ đầy ổ đĩa (crop body train ~9 GB).

In [ ]:
for f in glob.glob(f"{CACHE}/face_*"):
    os.remove(f)
sh(f"python scripts/train_reid.py {COMMON} --cls body --tag b_a --unfreeze 4 --lr-bb 2e-5 --epochs 10", cwd="manga-lens")
sh(f"python scripts/eval_cluster.py --chars {CH} --tag b_a --feat body --emb-dir {OUT} --no-test --json {OUT}/b_a_val.json", cwd="manga-lens")
sh(f"python scripts/train_reid.py {COMMON} --cls body --tag b_a --eval-only", cwd="manga-lens")
sh(f"python scripts/eval_cluster.py --chars {CH} --tag b_a --feat body --emb-dir {OUT} --json {OUT}/b_a_test.json", cwd="manga-lens")

## 8. Tổng kết + lưu kết quả
So với baseline chưa fine-tune (test, box GT): face BCubed F1 0,395 / ARI 0,131 / R@1 0,715; body 0,336 / 0,030 / 0,467.

In [ ]:
rows = [("face " + BEST_FACE, f"{OUT}/{BEST_FACE}_test.json"), ("body b_a", f"{OUT}/b_a_test.json")]
print(f"{'':14s} {'val bcF':>8s} {'test bcF':>9s} {'pairF':>6s} {'ARI':>6s} {'k_pred/k_true':>14s} {'R@1':>6s} {'mAP':>6s} {'shuffle':>8s}")
for name, f in rows:
    r = json.load(open(f))
    print(f"{name:14s} {r['val_bcF']:8.3f} {r['test_bcF']:9.3f} {r['test_pairF']:6.3f} {r['test_ARI']:6.3f} "
          f"{r['test_k_pred']:7.1f}/{r['test_k_true']:<6.1f} {r['test_R1']:6.3f} {r['test_mAP']:6.3f} {r['shuffle_bcF']:8.3f}")
shutil.rmtree("/kaggle/working/m109", ignore_errors=True)
shutil.rmtree("/kaggle/working/out/smoke", ignore_errors=True)
shutil.copy(CH, "/kaggle/working/out/characters.json")
sh("du -sh /kaggle/working/out/*; ls /kaggle/working/out/reid")